In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.cm as cm
from box_embeddings.parameterizations import MinDeltaBoxTensor
from box_embeddings.modules.intersection import GumbelIntersection
from box_embeddings.modules.volume import SoftVolume

# --- 1. SETUP ---
k = 5
num_dims = 2
feature_dim = 64
batch_size = 4

torch.manual_seed(42)
features = torch.randn(batch_size, feature_dim)

# Layer 1: Dalle Feature ai Box (4 coordinate)
projectors = nn.ModuleList([
    nn.Linear(in_features=feature_dim, out_features=2 * num_dims) for _ in range(k)
])

# Layer 2: Dai Box (4 coordinate) alla Probabilità (1 logit)
# QUESTO È IL PASSAGGIO CHIAVE! Dipende unicamente dalla geometria.
prob_predictors = nn.ModuleList([
    nn.Linear(in_features=2 * num_dims, out_features=1) for _ in range(k)
])

# Passiamo all'ottimizzatore i parametri di ENTRAMBI i gruppi di layer
optimizer = torch.optim.Adam(
    list(projectors.parameters()) + list(prob_predictors.parameters()), 
    lr=0.01
)

gumbel_intersection = GumbelIntersection(intersection_temperature=0.1)
soft_volume = SoftVolume(volume_temperature=0.5)

# --- 2. GROUND TRUTH ---
supervisions = [
    (0, 1, 1.0), 
    (0, 2, 1.0), 
    (1, 2, 0.0), 
    (2, 1, 0.0),
    (1, 3, 1.0), 
    (1, 4, 1.0), 
    (3, 4, 0.0), 
    (4, 3, 0.0),
]

concept_labels = torch.tensor([
    [1.0, 1.0, 0.0, 1.0, 0.0],
    [1.0, 0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0],
    [1.0, 1.0, 0.0, 0.0, 1.0],
], dtype=torch.float32)

AllenNLP not available. Registrable won't work.


In [2]:
epochs = 350 # Aumentiamo un po' le epoche data la maggiore complessità

for epoch in range(epochs):
    optimizer.zero_grad()
    
    boxes = []
    logits = []
    
    # --- FASE 1: Feature -> Box ---
    for i in range(k):
        theta_i = projectors[i](features) # (batch, 4)
        box_i = MinDeltaBoxTensor(theta_i.view(batch_size, 2, num_dims))
        boxes.append(box_i)
        
        # --- FASE 2: Box -> Probabilità ---
        # Estraiamo le coordinate reali z e Z che formano il box
        # Flattenizziamo le coordinate per darle in pasto al Linear (batch, 4)
        box_coords = torch.cat([box_i.z, box_i.Z], dim=-1) 
        
        # Il layer calcola la probabilità guardando SOLO la geometria del box
        logit_i = prob_predictors[i](box_coords).squeeze(-1) # (batch,)
        logits.append(logit_i)
        
    # --- CALCOLO DELLE LOSS ---
    
    # Loss 1: Attivazione dei Concetti (Basata sulla geometria)
    logits_tensor = torch.stack(logits, dim=1) # (batch, k)
    concept_probs = torch.sigmoid(logits_tensor)
    activation_loss = F.binary_cross_entropy(concept_probs, concept_labels)
    
    # Loss 2: Gerarchia (Intersezioni e Disgiunzioni)
    hierarchy_loss = 0.0
    for target_id, source_id, target_prob in supervisions:
        b_target = boxes[target_id]
        b_source = boxes[source_id]
        
        int_box = gumbel_intersection(b_target, b_source)
        log_vol_int = soft_volume(int_box)
        log_vol_source = soft_volume(b_source)
        
        pred_prob = torch.exp(log_vol_int - log_vol_source)
        pred_prob = torch.clamp(pred_prob, 1e-6, 1.0 - 1e-6)
        
        target_tensor = torch.full((batch_size,), target_prob, dtype=torch.float32)
        hierarchy_loss += F.binary_cross_entropy(pred_prob, target_tensor)
        
    # Loss 3: Anti-Collasso dei Volumi
    vol_loss = 0.0
    for i in range(1, k):
        vol_loss -= 0.1 * soft_volume(boxes[i]).mean()

    # Combinazione e Backpropagation
    final_loss = activation_loss + hierarchy_loss + vol_loss
    
    final_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoca {epoch+1} | Act: {activation_loss.item():.3f} | Hier: {hierarchy_loss.item():.3f}")

# Valutazione finale
with torch.no_grad():
    print("\n--- Probabilità di Attivazione (Calcolate dai Box) vs Ground Truth ---")
    final_probs = []
    for i in range(k):
        # Ripetiamo la cascata
        theta = projectors[i](features).view(batch_size, 2, num_dims)
        box = MinDeltaBoxTensor(theta)
        coords = torch.cat([box.z, box.Z], dim=-1)
        prob = torch.sigmoid(prob_predictors[i](coords).squeeze(-1))
        final_probs.append(prob)
        
    final_probs_tensor = torch.stack(final_probs, dim=1)
    
    for b in range(batch_size):
        print(f"Immagine {b}:")
        print(f"  Previste: {final_probs_tensor[b].numpy().round(2)}")
        print(f"  Reali:    {concept_labels[b].numpy()}")

Epoca 50 | Act: 0.337 | Hier: 0.072
Epoca 100 | Act: 0.104 | Hier: 0.093
Epoca 150 | Act: 0.034 | Hier: 0.063
Epoca 200 | Act: 0.015 | Hier: 0.048
Epoca 250 | Act: 0.009 | Hier: 0.040
Epoca 300 | Act: 0.006 | Hier: 0.034
Epoca 350 | Act: 0.004 | Hier: 0.030

--- Probabilità di Attivazione (Calcolate dai Box) vs Ground Truth ---
Immagine 0:
  Previste: [1.   1.   0.   0.99 0.  ]
  Reali:    [1. 1. 0. 1. 0.]
Immagine 1:
  Previste: [1.   0.01 0.99 0.01 0.  ]
  Reali:    [1. 0. 1. 0. 0.]
Immagine 2:
  Previste: [0.01 0.   0.   0.   0.  ]
  Reali:    [0. 0. 0. 0. 0.]
Immagine 3:
  Previste: [0.99 1.   0.01 0.   1.  ]
  Reali:    [1. 1. 0. 0. 1.]
